# 05 — Spatial Analysis: COVID-19

State-level analysis of confirmed cases, deaths, recovery, severity and first/second-wave burden. Uses the cleaned COVID dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("covid_clean.csv")
df = pd.read_csv(DATA_PATH)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.sort_values(["State/UnionTerritory", "Date"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("States:", df["State/UnionTerritory"].nunique())


## 1. Total cases, deaths and recoveries by state

In [ ]:
state_summary = (
    df.groupby("State/UnionTerritory", as_index=False)
      .agg(
          Confirmed=("Confirmed", "max"),
          Deaths=("Deaths", "max"),
          Cured=("Cured", "max")
      )
      .sort_values("Confirmed", ascending=False)
)

display(state_summary.head(15))

state_summary.head(15).set_index("State/UnionTerritory")["Confirmed"].sort_values().plot(kind="barh")
plt.title("Top States by Cumulative Confirmed Cases")
plt.xlabel("Confirmed cases")
plt.tight_layout()
plt.show()


## 2. Case Fatality Rate and Recovery Rate

In [ ]:
state_summary["CFR_%"] = state_summary["Deaths"] / state_summary["Confirmed"].replace(0, np.nan) * 100
state_summary["Recovery_Rate_%"] = state_summary["Cured"] / state_summary["Confirmed"].replace(0, np.nan) * 100

display(
    state_summary.sort_values("CFR_%", ascending=False)
    [["State/UnionTerritory", "Confirmed", "Deaths", "CFR_%", "Recovery_Rate_%"]]
    .head(15)
)


## 3. Spatial distribution of deaths

In [ ]:
state_summary.head(15).set_index("State/UnionTerritory")["Deaths"].sort_values().plot(kind="barh")
plt.title("Top States by Cumulative COVID-19 Deaths")
plt.xlabel("Deaths")
plt.tight_layout()
plt.show()


## 4. Peak daily cases by state

In [ ]:
if "NewCases" not in df.columns:
    df["NewCases"] = df.groupby("State/UnionTerritory")["Confirmed"].diff()

peak_cases = (
    df.groupby("State/UnionTerritory", as_index=False)
      .agg(Peak_NewCases=("NewCases", "max"))
      .sort_values("Peak_NewCases", ascending=False)
)

display(peak_cases.head(15))


## 5. Peak daily deaths by state

In [ ]:
if "NewDeaths" not in df.columns:
    df["NewDeaths"] = df.groupby("State/UnionTerritory")["Deaths"].diff()

peak_deaths = (
    df.groupby("State/UnionTerritory", as_index=False)
      .agg(Peak_NewDeaths=("NewDeaths", "max"))
      .sort_values("Peak_NewDeaths", ascending=False)
)

display(peak_deaths.head(15))


## 6. First-wave spatial burden

In [ ]:
first_wave = df[df["Date"] < pd.Timestamp("2021-01-01")]

first_wave_summary = (
    first_wave.groupby("State/UnionTerritory", as_index=False)
    .agg(Confirmed=("Confirmed", "max"), Deaths=("Deaths", "max"))
    .sort_values("Confirmed", ascending=False)
)

display(first_wave_summary.head(15))


## 7. Second-wave spatial burden

In [ ]:
second_wave = df[
    (df["Date"] >= pd.Timestamp("2021-01-01")) &
    (df["Date"] <= pd.Timestamp("2021-08-11"))
]

second_wave_summary = (
    second_wave.groupby("State/UnionTerritory", as_index=False)
    .agg(Confirmed=("Confirmed", "max"), Deaths=("Deaths", "max"))
    .sort_values("Confirmed", ascending=False)
)

display(second_wave_summary.head(15))


## 8. Contribution to India's reported cases

In [ ]:
national_total = state_summary["Confirmed"].sum()
state_summary["Share_of_state_sum_%"] = state_summary["Confirmed"] / national_total * 100

display(
    state_summary[["State/UnionTerritory", "Confirmed", "Share_of_state_sum_%"]]
    .head(15)
)


## 9. State severity comparison

In [ ]:
severity = state_summary.copy()
severity["Deaths_per_1000_cases"] = severity["Deaths"] / severity["Confirmed"].replace(0, np.nan) * 1000

display(
    severity.sort_values("Deaths_per_1000_cases", ascending=False)
    [["State/UnionTerritory", "Confirmed", "Deaths", "Deaths_per_1000_cases", "Recovery_Rate_%"]]
    .head(15)
)


## 10. Spatial analysis notes

In [ ]:
print("Key spatial-analysis cautions:")
print("1. Cumulative counts are strongly affected by population size.")
print("2. For population-adjusted comparisons, merge an external state population dataset.")
print("3. State names were cleaned earlier, so use the cleaned state field.")
print("4. Revisions can create unusual daily values; avoid interpreting negative revisions as real negative cases.")
print("5. Choropleth maps require a compatible India state GeoJSON/shapefile.")
